# Speech-Hands — Self-Reflection Demo

This notebook walks through **9 representative DCASE 2025 AudioQA samples** drawn from the paper's dev set. For each sample we show:

- The original multiple-choice question and its audio clip.
- What the **internal** (Qwen2.5-Omni-7B baseline) and **external** (Audio Flamingo 3) models predict on their own.
- The **self-reflection routing decision** our model makes — `<internal>` / `<external>` / `<rewrite>` — and its final answer.
- The gold label.

The samples are partitioned by routing decision so you can see each primitive in action:

| Count | Routing | When it's emitted |
|---|---|---|
| 3 | `<internal>` | Trust the model's own prediction, ignore external |
| 3 | `<external>` | Defer to the external model's top candidate |
| 3 | `<rewrite>`  | Neither is right; re-derive the answer |

> **Audio files** — this notebook expects DCASE 2025 AudioQA dev audio at `../data/dcase_audioqa/audio/dev/`. If you don't have it yet, the text-level routing/decision information is still fully displayed below; only the inline audio player will be silent.

In [ ]:
import json, os
from pathlib import Path
from IPython.display import Audio, HTML, Markdown, display

REPO_ROOT = Path(os.getcwd()).parent if Path(os.getcwd()).name == 'demo' else Path(os.getcwd())
CASES = json.load(open(REPO_ROOT / 'demo' / 'demo_cases.json'))
print(f'Loaded {len(CASES)} demo cases.')

In [ ]:
ROUTING_COLOR = {
    '<internal>': '#2a7a2a',
    '<external>': '#1a4fb8',
    '<rewrite>':  '#a83232',
}

def render_case(case, idx):
    audio_path = REPO_ROOT / case['audio']
    color = ROUTING_COLOR.get(case['speech_hands_decision'], '#333')
    header = f"### Case {idx+1} — {case['id']}  &middot;  decision: <span style='color:{color};font-weight:600;'>{case['speech_hands_decision']}</span>"
    body = (
        f"**Question:** {case['question']}\n\n"
        f"**Choices:**\n```\n{case['choices']}\n```\n"
        f"| Source | Prediction |\n|---|---|\n"
        f"| Qwen2.5-Omni-7B (internal) | `{case['qwen_baseline_pred']}` |\n"
        f"| Audio Flamingo 3 (external)| `{case['flamingo_external_pred']}` |\n"
        f"| **Speech-Hands final** | <span style='color:{color};font-weight:600;'>{case['speech_hands_final']}</span> |\n"
        f"| Gold | `{case['gold']}` |\n"
    )
    display(Markdown(header))
    if audio_path.exists():
        display(Audio(str(audio_path)))
    else:
        display(HTML(f"<em>Audio not available locally: <code>{case['audio']}</code></em>"))
    display(Markdown(body))
    display(HTML('<hr>'))

## 1. `<internal>` — trust the internal model

Cases where Qwen2.5-Omni alone already answers correctly. Speech-Hands learns to emit `<internal>` and echo the internal prediction instead of deferring to the (often noisier) external candidate.

In [ ]:
for i, c in enumerate([x for x in CASES if x['speech_hands_decision'] == '<internal>']):
    render_case(c, i)

## 2. `<external>` — defer to the external perception model

Cases where the internal model is confidently wrong but Audio Flamingo picks up the right option. Speech-Hands emits `<external>` and adopts Flamingo's answer.

In [ ]:
for i, c in enumerate([x for x in CASES if x['speech_hands_decision'] == '<external>']):
    render_case(c, i)

## 3. `<rewrite>` — neither candidate is right

The rarest routing. Both internal and external predictions are wrong, but with both audio and the external candidate list as grounding context the model re-derives the correct answer.

In [ ]:
for i, c in enumerate([x for x in CASES if x['speech_hands_decision'] == '<rewrite>']):
    render_case(c, i)

## Notes

- All 9 cases are drawn from the **DCASE 2025 AudioQA** dev set. The routing labels match the self-reflection SFT supervision files at the repo root (`internal_data.json`, `external_data.json`, `rewrite_data.json`).
- **No fine-tuned checkpoint is executed here** — we display recorded predictions from the paper's runs to illustrate the decision pattern. See `README.md` for running the full pipeline from code.
- To run the demo in full with your own inputs, train the Speech-Hands head with `configs/audio_qa/qwen2_5omni_full_sft_2025_with_audio_rag_v6.yaml`, then call `python test_qwen_2025.py <ckpt>` on your own DCASE manifest.